In [1]:
# ============================================================
# CELL 1 — Environment
# ============================================================

import sys
import torch
import transformers

print("=" * 70)
print("VehicleIQ — SigLIP 2 Text Retrieval Evaluation")
print("=" * 70)

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)

print("\nCUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU is not available. Reconnect to a GPU runtime."
    )

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)

props = torch.cuda.get_device_properties(0)

print(
    "VRAM:",
    round(props.total_memory / (1024 ** 3), 2),
    "GB"
)

VehicleIQ — SigLIP 2 Text Retrieval Evaluation
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
PyTorch: 2.11.0+cu128
Transformers: 5.13.1

CUDA available: True
GPU: Tesla T4
CUDA: 12.8
VRAM: 14.56 GB


In [2]:
# ============================================================
# CELL 2 — Mount Google Drive
# ============================================================

from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
# ============================================================
# CELL 3 — Locate VehicleIQ
# ============================================================

from pathlib import Path

MYDRIVE = Path(
    "/content/drive/MyDrive"
)

if not MYDRIVE.exists():
    raise RuntimeError(
        "MyDrive is not accessible."
    )

vehicleiq_candidates = [
    p for p in MYDRIVE.iterdir()
    if p.is_dir() and p.name.lower() == "vehicleiq"
]

print(
    "VehicleIQ candidates:",
    len(vehicleiq_candidates)
)

for p in vehicleiq_candidates:
    print(p)

if len(vehicleiq_candidates) != 1:
    raise RuntimeError(
        "Expected exactly one VehicleIQ folder."
    )

VEHICLIQ_ROOT = vehicleiq_candidates[0]

print(
    "\nUsing:",
    VEHICLIQ_ROOT
)

VehicleIQ candidates: 1
/content/drive/MyDrive/VehicleIQ

Using: /content/drive/MyDrive/VehicleIQ


In [5]:
# ============================================================
# CELL 5 — Exact VehicleIQ artifact paths
# ============================================================

DATASET_ROOT = (
    VEHICLIQ_ROOT
    / "datasets"
    / "AI_CITY_2023_TRACK2_NL_RETRIEVAL"
)

PROCESSED_ROOT = (
    DATASET_ROOT
    / "processed"
)

METADATA_ROOT = (
    PROCESSED_ROOT
    / "metadata"
)

EMBEDDING_ROOT = (
    PROCESSED_ROOT
    / "embeddings"
    / "siglip2"
)

# ------------------------------------------------------------
# Exact files confirmed from Google Drive
# ------------------------------------------------------------

EMBEDDINGS_PATH = (
    EMBEDDING_ROOT
    / "vehicle_embeddings.npy"
)

VEHICLE_IDS_PATH = (
    EMBEDDING_ROOT
    / "vehicle_ids.json"
)

FAISS_PATH = (
    EMBEDDING_ROOT
    / "vehicle_gallery.faiss"
)

SAMPLED_METADATA_PATH = (
    METADATA_ROOT
    / "train_sampled_8.json"
)

# ------------------------------------------------------------
# Verify
# ------------------------------------------------------------

print("Embeddings:")
print(EMBEDDINGS_PATH)

print("\nVehicle IDs:")
print(VEHICLE_IDS_PATH)

print("\nFAISS:")
print(FAISS_PATH)

print("\nSampled metadata:")
print(SAMPLED_METADATA_PATH)

print("\nExistence checks:")

print(
    "vehicle_embeddings.npy:",
    EMBEDDINGS_PATH.exists()
)

print(
    "vehicle_ids.json:",
    VEHICLE_IDS_PATH.exists()
)

print(
    "vehicle_gallery.faiss:",
    FAISS_PATH.exists()
)

print(
    "train_sampled_8.json:",
    SAMPLED_METADATA_PATH.exists()
)

assert EMBEDDINGS_PATH.exists()
assert VEHICLE_IDS_PATH.exists()
assert FAISS_PATH.exists()
assert SAMPLED_METADATA_PATH.exists()

print(
    "\n✓ All required VehicleIQ artifacts found."
)

Embeddings:
/content/drive/MyDrive/VehicleIQ/datasets/AI_CITY_2023_TRACK2_NL_RETRIEVAL/processed/embeddings/siglip2/vehicle_embeddings.npy

Vehicle IDs:
/content/drive/MyDrive/VehicleIQ/datasets/AI_CITY_2023_TRACK2_NL_RETRIEVAL/processed/embeddings/siglip2/vehicle_ids.json

FAISS:
/content/drive/MyDrive/VehicleIQ/datasets/AI_CITY_2023_TRACK2_NL_RETRIEVAL/processed/embeddings/siglip2/vehicle_gallery.faiss

Sampled metadata:
/content/drive/MyDrive/VehicleIQ/datasets/AI_CITY_2023_TRACK2_NL_RETRIEVAL/processed/metadata/train_sampled_8.json

Existence checks:
vehicle_embeddings.npy: True
vehicle_ids.json: True
vehicle_gallery.faiss: True
train_sampled_8.json: True

✓ All required VehicleIQ artifacts found.


In [8]:
!pip install faiss-gpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.2/135.2 MB 12.0 MB/s eta 0:00:00


In [9]:
# ============================================================
# CELL 6 — Load existing image gallery
# ============================================================

import numpy as np
import json
import faiss

vehicle_embeddings = np.load(
    EMBEDDINGS_PATH
)

with open(
    VEHICLE_IDS_PATH,
    "r",
    encoding="utf-8"
) as f:
    vehicle_ids = json.load(f)

index = faiss.read_index(
    str(FAISS_PATH)
)

print(
    "Embedding matrix:",
    vehicle_embeddings.shape
)

print(
    "Vehicle IDs:",
    len(vehicle_ids)
)

print(
    "FAISS vectors:",
    index.ntotal
)

print(
    "FAISS dimension:",
    index.d
)

print(
    "Embedding dtype:",
    vehicle_embeddings.dtype
)

# ----------------------------
# Integrity checks
# ----------------------------

assert vehicle_embeddings.shape == (
    2155,
    768
)

assert len(vehicle_ids) == 2155

assert index.ntotal == 2155

assert index.d == 768

assert len(set(vehicle_ids)) == 2155

norms = np.linalg.norm(
    vehicle_embeddings,
    axis=1
)

print(
    "\nMin embedding norm:",
    norms.min()
)

print(
    "Max embedding norm:",
    norms.max()
)

print("\n✓ Existing gallery validated.")

Embedding matrix: (2155, 768)
Vehicle IDs: 2155
FAISS vectors: 2155
FAISS dimension: 768
Embedding dtype: float32

Min embedding norm: 0.9999998
Max embedding norm: 1.0000001

✓ Existing gallery validated.


In [10]:
# ============================================================
# CELL 7 — Load evaluation metadata
# ============================================================

with open(
    SAMPLED_METADATA_PATH,
    "r",
    encoding="utf-8"
) as f:
    evaluation_metadata = json.load(f)

print(
    "Vehicles in metadata:",
    len(evaluation_metadata)
)

first_vehicle_id = next(
    iter(evaluation_metadata)
)

first_record = evaluation_metadata[
    first_vehicle_id
]

print(
    "\nFirst vehicle:",
    first_vehicle_id
)

print(
    "\nRecord keys:",
    first_record.keys()
)

print(
    "\nDescriptions:"
)

for description in first_record[
    "descriptions"
]:
    print("-", description)

Vehicles in metadata: 2155

First vehicle: b06c903c-a25d-45fe-b0d5-294f72e34023

Record keys: dict_keys(['vehicle_id', 'split', 'source', 'images', 'descriptions', 'descriptions_other_views', 'sampling'])

Descriptions:
- A red sedan drives forward.
- A red midsize sedan keep straight.
- A red car drove through an intersection.


In [11]:
# ============================================================
# CELL 8 — Build evaluation queries
# ============================================================

evaluation_queries = []

for vehicle_id, record in (
    evaluation_metadata.items()
):

    for description in record.get(
        "descriptions",
        []
    ):

        if not description:
            continue

        evaluation_queries.append({
            "query": description,
            "vehicle_id": vehicle_id
        })

print(
    "Evaluation queries:",
    len(evaluation_queries)
)

print(
    "Vehicles represented:",
    len(
        set(
            item["vehicle_id"]
            for item in evaluation_queries
        )
    )
)

assert len(evaluation_queries) == 6465
assert len(
    set(
        item["vehicle_id"]
        for item in evaluation_queries
    )
) == 2155

print("\n✓ Evaluation set created.")

Evaluation queries: 6465
Vehicles represented: 2155

✓ Evaluation set created.


In [12]:
# ============================================================
# CELL 9 — Verify evaluation alignment
# ============================================================

gallery_ids = set(vehicle_ids)

evaluation_vehicle_ids = set(
    item["vehicle_id"]
    for item in evaluation_queries
)

missing_ids = (
    evaluation_vehicle_ids
    - gallery_ids
)

print(
    "Evaluation vehicles:",
    len(evaluation_vehicle_ids)
)

print(
    "Missing from gallery:",
    len(missing_ids)
)

assert len(missing_ids) == 0

print(
    "\n✓ Every evaluation vehicle exists in gallery."
)

Evaluation vehicles: 2155
Missing from gallery: 0

✓ Every evaluation vehicle exists in gallery.


In [13]:
# ============================================================
# CELL 10 — Load pretrained SigLIP 2
# ============================================================

from transformers import (
    AutoProcessor,
    AutoModel
)

MODEL_ID = (
    "google/siglip2-base-patch16-224"
)

device = torch.device("cuda")

print(
    "Loading:",
    MODEL_ID
)

processor = AutoProcessor.from_pretrained(
    MODEL_ID
)

model = AutoModel.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16
).to(device)

model.eval()

print(
    "\nModel class:"
)

print(
    type(model)
)

print(
    "\nModel path:"
)

print(
    model.name_or_path
)

print(
    "\nConfig model_type:"
)

print(
    getattr(
        model.config,
        "model_type",
        None
    )
)

Loading: google/siglip2-base-patch16-224


preprocessor_config.json:   0%|          | 0.00/394 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/253 [00:00<?, ?B/s]

[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.


tokenizer_config.json:   0%|          | 0.00/47.2k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 34.4MB            

... (3 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.50GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]


Model class:
<class 'transformers.models.siglip.modeling_siglip.SiglipModel'>

Model path:
google/siglip2-base-patch16-224

Config model_type:
siglip


In [14]:
# ============================================================
# CELL 11 — Native SigLIP sanity check
# ============================================================

from PIL import Image

test_vehicle_id = first_vehicle_id

test_record = evaluation_metadata[
    test_vehicle_id
]

test_image_info = (
    test_record["images"][0]
)

test_image_path = (
    VEHICLIQ_ROOT
)

# Search for the actual crop path under VehicleIQ
crop_matches = list(
    VEHICLIQ_ROOT.rglob(
        Path(
            test_image_info["crop_path"]
        ).name
    )
)

print(
    "Candidate crop files found:",
    len(crop_matches)
)

if len(crop_matches) == 0:
    raise FileNotFoundError(
        "Could not locate representative crop."
    )

# Prefer the crop whose path contains the vehicle ID
preferred = [
    p for p in crop_matches
    if test_vehicle_id in str(p)
]

if len(preferred) == 1:
    test_image_path = preferred[0]

elif len(crop_matches) == 1:
    test_image_path = crop_matches[0]

else:
    raise RuntimeError(
        "Could not uniquely identify test crop."
    )

test_image = Image.open(
    test_image_path
).convert("RGB")

test_text = (
    test_record["descriptions"][0]
)

print(
    "Test vehicle:",
    test_vehicle_id
)

print(
    "Test text:",
    test_text
)

print(
    "Image:",
    test_image_path
)

# ------------------------------------------------------------
# Native model
# ------------------------------------------------------------

inputs = processor(
    text=[test_text],
    images=[test_image],
    padding="max_length",
    return_tensors="pt"
)

inputs = {
    key: value.to(device)
    for key, value in inputs.items()
    if torch.is_tensor(value)
}

with torch.inference_mode():

    outputs = model(
        **inputs
    )

print(
    "\nImage embeds:",
    outputs.image_embeds.shape
)

print(
    "Text embeds:",
    outputs.text_embeds.shape
)

print(
    "Image norm:",
    outputs.image_embeds.norm(
        dim=-1
    ).item()
)

print(
    "Text norm:",
    outputs.text_embeds.norm(
        dim=-1
    ).item()
)

print(
    "Native image-text logit:",
    outputs.logits_per_image.item()
)

Candidate crop files found: 60
Test vehicle: b06c903c-a25d-45fe-b0d5-294f72e34023
Test text: A red sedan drives forward.
Image: /content/drive/MyDrive/VehicleIQ/datasets/AI_CITY_2023_TRACK2_NL_RETRIEVAL/processed/train/tracks/b06c903c-a25d-45fe-b0d5-294f72e34023/frame_000001.jpg

Image embeds: torch.Size([1, 768])
Text embeds: torch.Size([1, 768])
Image norm: 1.0
Text norm: 1.0
Native image-text logit: -0.109375


In [15]:
# ============================================================
# CELL 12 — Text encoder
# ============================================================

import torch.nn.functional as F

def encode_text(
    texts,
    batch_size=32
):
    """
    Text -> normalized 768-D SigLIP embedding.
    """

    if isinstance(texts, str):
        texts = [texts]

    embeddings_list = []

    for start in range(
        0,
        len(texts),
        batch_size
    ):

        batch_texts = texts[
            start:start + batch_size
        ]

        inputs = processor(
            text=batch_texts,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        inputs = {
            key: value.to(device)
            for key, value in inputs.items()
            if torch.is_tensor(value)
        }

        with torch.inference_mode():

            output = model.get_text_features(
                **inputs
            )

        if hasattr(
            output,
            "pooler_output"
        ):
            batch_embeddings = (
                output.pooler_output
            )
        else:
            batch_embeddings = output

        batch_embeddings = F.normalize(
            batch_embeddings,
            p=2,
            dim=-1
        )

        embeddings_list.append(
            batch_embeddings
        )

    return torch.cat(
        embeddings_list,
        dim=0
    )

In [16]:
# ============================================================
# CELL 14 — Single-query retrieval
# ============================================================

query = (
    "A red sedan drives forward."
)

query_embedding = encode_text(
    query
)

query_vector = (
    query_embedding
    .detach()
    .cpu()
    .numpy()
    .astype(np.float32)
)

scores, indices = index.search(
    query_vector,
    10
)

retrieved_ids = [
    vehicle_ids[idx]
    for idx in indices[0]
]

print("Query:")
print(query)

print("\nTop 10:")

for rank, (
    vehicle_id,
    score
) in enumerate(
    zip(
        retrieved_ids,
        scores[0]
    ),
    start=1
):

    print(
        f"{rank:2d}. "
        f"{vehicle_id} "
        f"similarity={score:.4f}"
    )

ground_truth = (
    evaluation_queries[0]["vehicle_id"]
)

print(
    "\nGround truth:",
    ground_truth
)

if ground_truth in retrieved_ids:

    print(
        "Ground-truth rank:",
        retrieved_ids.index(
            ground_truth
        ) + 1
    )

else:

    print(
        "Ground truth: NOT TOP-10"
    )

Query:
A red sedan drives forward.

Top 10:
 1. eb184f7c-35c7-4f3e-af52-1db57fb087d4 similarity=0.1703
 2. cfad5a2f-3aba-450a-8c80-3d76d1dadf62 similarity=0.1699
 3. 0ee59af3-1502-494b-a3a3-1c4fade7873b similarity=0.1675
 4. 1a9355dc-9ccc-4f12-a853-1bb79e4e5e0e similarity=0.1665
 5. 3a02a86d-154b-4ee2-bd5e-a0811113a9d8 similarity=0.1663
 6. 4ad0b436-0bb9-4a2b-a3f7-7b74d3fd8668 similarity=0.1658
 7. b994a201-f6da-4e9c-b5cd-85f667aa816b similarity=0.1657
 8. 610b30a1-5f40-41d0-9c05-fbd1448d3f6e similarity=0.1654
 9. 870e5570-4c79-42ba-bd97-a862d86004f0 similarity=0.1654
10. ca661efe-d817-47f7-a699-3d7e15c64f33 similarity=0.1652

Ground truth: b06c903c-a25d-45fe-b0d5-294f72e34023
Ground truth: NOT TOP-10


In [17]:
# ============================================================
# CELL 15 — Practical semantic retrieval sanity test
# ============================================================

test_queries = [
    "red sedan",
    "white pickup truck",
    "blue pickup truck",
    "white truck",
    "blue bus",
    "motorcycle"
]

for query in test_queries:

    query_embedding = encode_text(
        query
    )

    query_vector = (
        query_embedding
        .detach()
        .cpu()
        .numpy()
        .astype(np.float32)
    )

    scores, indices = index.search(
        query_vector,
        10
    )

    print("\n" + "=" * 80)
    print("QUERY:", query)
    print("=" * 80)

    for rank, (score, idx) in enumerate(
        zip(scores[0], indices[0]),
        start=1
    ):

        vehicle_id = vehicle_ids[idx]

        record = evaluation_metadata[
            vehicle_id
        ]

        descriptions = record.get(
            "descriptions",
            []
        )

        print(
            f"\n#{rank} "
            f"score={score:.4f}"
        )

        print(
            "Vehicle:",
            vehicle_id
        )

        for description in descriptions:
            print(
                "  -",
                description
            )


QUERY: red sedan

#1 score=0.1521
Vehicle: b1f099fd-e45e-4ded-b183-725741959fba
  - A small red sedan turns right at an intersection.
  - A small maroon sedan takes a right at an intersection after a maroon suv turns right.
  - Red car turning right.

#2 score=0.1497
Vehicle: b994a201-f6da-4e9c-b5cd-85f667aa816b
  - A red sedan keeps straight across the intersection.
  - A red sedan runs through intersection.
  - A red sedan runs across the intersection.

#3 score=0.1474
Vehicle: 610b30a1-5f40-41d0-9c05-fbd1448d3f6e
  - A red car drives behind a larger car.
  - A red sedan stops at the intersection  after another gray vehicle.
  - A maroon sedan closely follows another vehicle.

#4 score=0.1472
Vehicle: f70f8c5d-bdab-41f5-8f04-a4b7915d1a06
  - A red sedan turns left.
  - A red sedan turns slightly left at an intersection.
  - A red sedan runs down a busy street with many vehicles parking aside and many vehicles follow and pass by on the opposite direction.

#5 score=0.1471
Vehicle: 84

In [18]:
# ============================================================
# CELL 16 — Inspect vehicle category coverage
# ============================================================

from collections import Counter

vehicle_category_counts = Counter()

for vehicle_id, record in evaluation_metadata.items():

    descriptions = record.get(
        "descriptions",
        []
    )

    text = " ".join(
        descriptions
    ).lower()

    if "motorcycle" in text or "motorbike" in text:
        category = "motorcycle"

    elif "bus" in text:
        category = "bus"

    elif "pickup" in text:
        category = "pickup"

    elif "truck" in text:
        category = "truck"

    elif "sedan" in text:
        category = "sedan"

    elif "suv" in text:
        category = "suv"

    elif "van" in text:
        category = "van"

    elif "jeep" in text:
        category = "jeep"

    else:
        category = "other"

    vehicle_category_counts[
        category
    ] += 1

print(
    "Vehicle category coverage:"
)

for category, count in (
    vehicle_category_counts
    .most_common()
):

    print(
        f"{category:12s}: {count}"
    )

Vehicle category coverage:
suv         : 751
sedan       : 748
pickup      : 371
bus         : 145
truck       : 72
van         : 43
other       : 19
jeep        : 6


In [20]:
# ============================================================
# CELL 16 — Semantic relevance definitions
# ============================================================

semantic_queries = {
    "red sedan": {
        "required": ["red", "sedan"]
    },

    "white pickup truck": {
        "required": ["white", "pickup"]
    },

    "blue pickup truck": {
        "required": ["blue", "pickup"]
    },

    "white truck": {
        "required": ["white", "truck"]
    },

    "blue bus": {
        "required": ["blue", "bus"]
    },

    "red SUV": {
        "required": ["red", "suv"]
    },

    "black SUV": {
        "required": ["black", "suv"]
    },

    "white van": {
        "required": ["white", "van"]
    }
}

print("Semantic benchmark queries:\n")

for query, spec in semantic_queries.items():
    print(
        f"{query:25s} -> {spec['required']}"
    )

Semantic benchmark queries:

red sedan                 -> ['red', 'sedan']
white pickup truck        -> ['white', 'pickup']
blue pickup truck         -> ['blue', 'pickup']
white truck               -> ['white', 'truck']
blue bus                  -> ['blue', 'bus']
red SUV                   -> ['red', 'suv']
black SUV                 -> ['black', 'suv']
white van                 -> ['white', 'van']


In [21]:
# ============================================================
# CELL 17 — Build searchable vehicle descriptions
# ============================================================

vehicle_search_text = {}

for vehicle_id, record in evaluation_metadata.items():

    descriptions = record.get(
        "descriptions",
        []
    )

    text = " ".join(
        descriptions
    ).lower()

    vehicle_search_text[
        vehicle_id
    ] = text

print(
    "Vehicles indexed:",
    len(vehicle_search_text)
)

print(
    "\nExample:"
)

example_id = first_vehicle_id

print(
    example_id
)

print(
    vehicle_search_text[
        example_id
    ]
)

Vehicles indexed: 2155

Example:
b06c903c-a25d-45fe-b0d5-294f72e34023
a red sedan drives forward. a red midsize sedan keep straight. a red car drove through an intersection.


In [22]:
# ============================================================
# CELL 18 — Semantic relevance function
# ============================================================

def is_semantically_relevant(
    vehicle_id,
    required_terms
):
    """
    Baseline semantic relevance rule.

    A vehicle is relevant when every required
    concept appears somewhere in its descriptions.
    """

    text = vehicle_search_text[
        vehicle_id
    ]

    return all(
        term.lower() in text
        for term in required_terms
    )

In [23]:
# ============================================================
# CELL 19 — Relevance-label sanity check
# ============================================================

for query, spec in semantic_queries.items():

    relevant_count = sum(
        is_semantically_relevant(
            vehicle_id,
            spec["required"]
        )
        for vehicle_id in vehicle_ids
    )

    print(
        f"{query:25s} -> "
        f"{relevant_count} relevant vehicles"
    )

red sedan                 -> 209 relevant vehicles
white pickup truck        -> 149 relevant vehicles
blue pickup truck         -> 57 relevant vehicles
white truck               -> 184 relevant vehicles
blue bus                  -> 13 relevant vehicles
red SUV                   -> 277 relevant vehicles
black SUV                 -> 495 relevant vehicles
white van                 -> 147 relevant vehicles


In [24]:
# ============================================================
# CELL 20 — Semantic retrieval
# ============================================================

def semantic_search(
    query,
    top_k=10
):

    query_embedding = encode_text(
        query
    )

    query_vector = (
        query_embedding
        .detach()
        .cpu()
        .numpy()
        .astype(np.float32)
    )

    scores, indices = index.search(
        query_vector,
        top_k
    )

    results = []

    for rank, (
        score,
        idx
    ) in enumerate(
        zip(
            scores[0],
            indices[0]
        ),
        start=1
    ):

        vehicle_id = vehicle_ids[idx]

        results.append({
            "rank": rank,
            "vehicle_id": vehicle_id,
            "similarity": float(score)
        })

    return results

In [25]:
# ============================================================
# CELL 21 — Single semantic-query evaluation
# ============================================================

query = "red sedan"

spec = semantic_queries[
    query
]

results = semantic_search(
    query,
    top_k=10
)

print("=" * 70)
print("QUERY:", query)
print("=" * 70)

relevant_count = 0

for result in results:

    relevant = is_semantically_relevant(
        result["vehicle_id"],
        spec["required"]
    )

    if relevant:
        relevant_count += 1

    print(
        f"#{result['rank']:2d} "
        f"score={result['similarity']:.4f} "
        f"relevant={relevant}"
    )

print(
    "\nRelevant results in Top-10:",
    relevant_count
)

print(
    "Precision@10:",
    relevant_count / 10
)

QUERY: red sedan
# 1 score=0.1521 relevant=True
# 2 score=0.1497 relevant=True
# 3 score=0.1474 relevant=True
# 4 score=0.1472 relevant=True
# 5 score=0.1471 relevant=True
# 6 score=0.1467 relevant=True
# 7 score=0.1465 relevant=True
# 8 score=0.1462 relevant=True
# 9 score=0.1459 relevant=True
#10 score=0.1454 relevant=True

Relevant results in Top-10: 10
Precision@10: 1.0


In [26]:
# ============================================================
# CELL 22 — Semantic retrieval benchmark
# ============================================================

benchmark_results = {}

for query, spec in semantic_queries.items():

    results = semantic_search(
        query,
        top_k=10
    )

    relevance = [
        is_semantically_relevant(
            result["vehicle_id"],
            spec["required"]
        )
        for result in results
    ]

    precision_at_10 = (
        sum(relevance) / 10.0
    )

    benchmark_results[
        query
    ] = {
        "results": results,
        "relevance": relevance,
        "precision_at_10": precision_at_10
    }

    print(
        f"{query:25s} "
        f"P@10={precision_at_10:.2f}"
    )

red sedan                 P@10=1.00
white pickup truck        P@10=1.00
blue pickup truck         P@10=1.00
white truck               P@10=0.90
blue bus                  P@10=0.10
red SUV                   P@10=0.50
black SUV                 P@10=0.70
white van                 P@10=1.00


In [27]:
# ============================================================
# CELL 23 — Aggregate baseline metric
# ============================================================

precision_values = [
    result["precision_at_10"]
    for result in benchmark_results.values()
]

mean_precision_at_10 = np.mean(
    precision_values
)

print(
    "Mean Precision@10:",
    round(
        mean_precision_at_10,
        4
    )
)

print(
    "Mean Precision@10 (%):",
    round(
        mean_precision_at_10 * 100,
        2
    ),
    "%"
)

Mean Precision@10: 0.775
Mean Precision@10 (%): 77.5 %


In [28]:
# ============================================================
# CELL 24 — NDCG@10
# ============================================================

import math

def dcg_at_k(
    relevance,
    k=10
):

    score = 0.0

    for rank, rel in enumerate(
        relevance[:k],
        start=1
    ):

        if rel:
            score += (
                1.0 /
                math.log2(rank + 1)
            )

    return score


def ndcg_at_k(
    relevance,
    k=10
):

    actual_dcg = dcg_at_k(
        relevance,
        k
    )

    ideal_relevance = sorted(
        relevance,
        reverse=True
    )

    ideal_dcg = dcg_at_k(
        ideal_relevance,
        k
    )

    if ideal_dcg == 0:
        return 0.0

    return actual_dcg / ideal_dcg


ndcg_values = []

for query, result in (
    benchmark_results.items()
):

    ndcg = ndcg_at_k(
        result["relevance"],
        k=10
    )

    result["ndcg_at_10"] = ndcg

    ndcg_values.append(
        ndcg
    )

mean_ndcg_at_10 = np.mean(
    ndcg_values
)

print(
    "Mean NDCG@10:",
    round(
        mean_ndcg_at_10,
        4
    )
)

print(
    "Mean NDCG@10 (%):",
    round(
        mean_ndcg_at_10 * 100,
        2
    ),
    "%"
)

Mean NDCG@10: 0.8878
Mean NDCG@10 (%): 88.78 %


In [29]:
# ============================================================
# CELL 25 — Final pretrained baseline report
# ============================================================

print("=" * 70)
print("VehicleIQ — Pretrained SigLIP 2 Semantic Retrieval Baseline")
print("=" * 70)

print(
    "Gallery size:",
    len(vehicle_ids)
)

print(
    "Embedding dimension:",
    index.d
)

print(
    "Benchmark queries:",
    len(semantic_queries)
)

print()

print(
    f"Mean Precision@10: "
    f"{mean_precision_at_10 * 100:.2f}%"
)

print(
    f"Mean NDCG@10:      "
    f"{mean_ndcg_at_10 * 100:.2f}%"
)

print()

for query, result in (
    benchmark_results.items()
):

    print(
        f"{query:25s}"
        f"P@10={result['precision_at_10']:.2f} "
        f"NDCG@10={result['ndcg_at_10']:.2f}"
    )

print("=" * 70)

VehicleIQ — Pretrained SigLIP 2 Semantic Retrieval Baseline
Gallery size: 2155
Embedding dimension: 768
Benchmark queries: 8

Mean Precision@10: 77.50%
Mean NDCG@10:      88.78%

red sedan                P@10=1.00 NDCG@10=1.00
white pickup truck       P@10=1.00 NDCG@10=1.00
blue pickup truck        P@10=1.00 NDCG@10=1.00
white truck              P@10=0.90 NDCG@10=0.95
blue bus                 P@10=0.10 NDCG@10=0.39
red SUV                  P@10=0.50 NDCG@10=0.85
black SUV                P@10=0.70 NDCG@10=0.91
white van                P@10=1.00 NDCG@10=1.00


In [32]:
# ============================================================
# CELL 26 — Save baseline results
# ============================================================

from pathlib import Path
import json

# ------------------------------------------------------------
# Define evaluation output directory
# ------------------------------------------------------------

EVALUATION_ROOT = (
    PROCESSED_ROOT
    / "evaluation"
    / "siglip2_text_retrieval"
)

EVALUATION_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# Build baseline record
# ------------------------------------------------------------

baseline_output = {
    "model": "google/siglip2-base-patch16-224",

    "gallery_size": len(vehicle_ids),

    "embedding_dimension": int(
        index.d
    ),

    "benchmark_queries": semantic_queries,

    "mean_precision_at_10": float(
        mean_precision_at_10
    ),

    "mean_ndcg_at_10": float(
        mean_ndcg_at_10
    ),

    "query_results": benchmark_results
}

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

OUTPUT_PATH = (
    EVALUATION_ROOT
    / "siglip2_semantic_baseline.json"
)

with open(
    OUTPUT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        baseline_output,
        f,
        indent=2
    )

print(
    "Saved baseline:"
)

print(
    OUTPUT_PATH
)

Saved baseline:
/content/drive/MyDrive/VehicleIQ/datasets/AI_CITY_2023_TRACK2_NL_RETRIEVAL/processed/evaluation/siglip2_text_retrieval/siglip2_semantic_baseline.json
